# Splunk draw.io icons — tutorial

This notebook walks through **how to use** the tooling in this repo. The implementation lives in Python modules (`splunk_icons_pipeline.py`); you do not need to run giant code cells here.

**Fastest path:** skip to section **5. Import into draw.io** and open the prebuilt files under `dist/`.

## What you get

| Output | Use when |
|--------|----------|
| `dist/Splunk-Icons-adaptive.xml` | Light **and** dark diagrams (theme-aware silhouettes) — **recommended** |
| `dist/Splunk-Icons-color.xml` | Full-color icons on light backgrounds |
| `dist/Splunk-Icons-dark.xml` | Inverted icons on dark backgrounds |
| `dist/Splunk-Connectors.xml` | Splunk-style connector line presets |

Import the **`.xml`** files (not `.drawiolib`) via **File → Open Library From → Device** in [diagrams.net](https://app.diagrams.net/).

## 1. Setup (one time)

From the repo root, with Python 3.10+:

```bash
python3 -m venv .venv
source .venv/bin/activate    # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

Install **Tesseract OCR** (needed only if you re-run OCR):

- macOS: `brew install tesseract`
- Ubuntu: `sudo apt install tesseract-ocr`

Download Splunk's official sheet and save it as:

`source/Splunk_Documentation_Icons_August2018.png`

(Splunk docs: [Draw a diagram of your deployment](https://help.splunk.com/en/splunk-enterprise/administer/inherit-a-splunk-deployment/10.4/inherited-deployment-tasks/draw-a-diagram-of-your-deployment).)

In [ ]:
# Confirm the environment (run from repo root after activating .venv)
from pathlib import Path
root = Path('.').resolve()
print('Repo:', root)
print('Source PNG:', (root / 'source' / 'Splunk_Documentation_Icons_August2018.png').exists())
print('Prebuilt adaptive library:', (root / 'dist' / 'Splunk-Icons-adaptive.xml').exists())

## 2. Rebuild everything (CLI)

The pipeline is a script — no notebook required:

```bash
python run_pipeline.py all
```

Individual steps (each writes files under `dist/`):

| Command | What it does |
|---------|----------------|
| `python run_pipeline.py crops` | Detect icons, write `dist/crops/`, `manifest.json`, `connectors.json` |
| `python run_pipeline.py ocr` | Tesseract on label bands → `labels_ocr.json` |
| `python run_pipeline.py labels` | Fuzzy title match → `labels_final.json` |
| `python run_pipeline.py build` | Generate `Splunk-Icons-*.xml` and connectors |

In [ ]:
# Uncomment to rebuild libraries from disk:
# !python run_pipeline.py build

## 3. Fix titles in a browser

```bash
python edit_labels.py
```

Open http://127.0.0.1:8765 — search icons, edit titles, mark complete, save. Then rebuild:

```bash
python edit_labels.py --rebuild
```

(or `python run_pipeline.py build`)

## 4. Add custom icon crops

When an icon is missing from the auto grid, draw a rectangle on the source sheet:

```bash
python crop_picker.py
```

Open http://127.0.0.1:8766, drag a box, save crop, set a title. Re-run **`python run_pipeline.py build`** so libraries include custom entries from `dist/custom_crops.json`.

## 5. Import into draw.io

1. Open [diagrams.net](https://app.diagrams.net/) or the desktop app.
2. **File → Open Library From → Device** (not *Import* — that opens diagrams).
3. Choose e.g. `dist/Splunk-Icons-adaptive.xml`.
4. Enable the library in the left sidebar if needed (**+** / **More Shapes**).

Repeat for `Splunk-Connectors.xml` if you want connector styles.

## 6. Explore from Python (optional)

You can call pipeline functions directly — useful in this notebook:

In [ ]:
from pathlib import Path
import json
from splunk_icons_pipeline import DIST_DIR, step_build

labels = json.loads((DIST_DIR / 'labels_final.json').read_text())
connectors = json.loads((DIST_DIR / 'connectors.json').read_text())
step_build(labels, connectors)
print('Wrote libraries under', DIST_DIR)

## Legal

Splunk icon artwork belongs to Splunk. This repo includes **generated** libraries in `dist/` for convenience; obtain the source PNG from Splunk documentation and comply with Splunk's terms.